# 03 — Baseline Segmentation Model Training

**Project:** Obstetric Ultrasound Segmentation and Fetal Head Circumference (HC) Measurement (SBES426 — Team 2)

## Objectives
1. Load leakage-free Train / Validation manifests from `data/processed/processed_metadata.csv`.
2. Mitigate **trimester/size distribution imbalance** via tempered HC-stratum sampling and training-only geometric/photometric augmentations (flips, ±15° rotation, mild scale/shift, contrast/brightness jitter, subtle Gaussian noise).
3. Mitigate **pixel-level foreground/background imbalance** (~30.6% head vs. 69.4% background) via combined **Soft Dice Loss + Weighted BCE** (`pos_weight` computed strictly on training masks).
4. Train the identical lightweight **Baseline U-Net** (`491,729` parameters, 4 encoder/decoder stages: `8 -> 16 -> 32 -> 64 -> 128`) on **both** `raw_norm` (control) and `despeckled` (Perona-Malik) input arms, saving the best validation checkpoints and learning curves.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import torch
from IPython.display import Image, display

from src.config import PROCESSED_DATA_DIR, REPORTS_DIR, RANDOM_SEED
from src.models.dataset import build_dataloaders
from src.models.unet import BaselineUNet, count_parameters
from src.models.trainer import (
    TrainConfig,
    train_baseline_model,
    plot_training_curves,
    plot_validation_predictions,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Inspect DataLoaders, Class Weight & Model Architecture
Verify batch tensor shapes, positive pixel class weight (`pos_weight`), and trainable parameter count.

In [ ]:
loaders, split_dfs, pos_weight = build_dataloaders(
    image_arm="raw_norm",
    batch_size=16,
    input_size=(128, 192),
    use_stratum_sampler=True,
    seed=RANDOM_SEED,
)
sample_batch = next(iter(loaders["train"]))
print("Train batches:", len(loaders["train"]), "| Val batches:", len(loaders["val"]), "| Test batches:", len(loaders["test"]))
print("Batch image shape:", tuple(sample_batch["image"].shape), "| mask shape:", tuple(sample_batch["mask"].shape))
print(f"Training-only BCE pos_weight: {pos_weight:.3f}")

demo_model = BaselineUNet(in_channels=1, out_channels=1, base_channels=8, bottleneck_dropout=0.15)
print(f"BaselineUNet trainable parameters: {count_parameters(demo_model):,}")

## 2. Train Baseline U-Net on Both Arms (`raw_norm` vs. `despeckled`)
Both arms share identical initialization seed, architecture, optimizer (`AdamW`, `lr=2e-3`, `weight_decay=1e-4`), `CosineAnnealingLR` schedule, augmentations, and `DiceBCELoss` so any performance difference isolates the effect of edge-preserving speckle reduction.

In [ ]:
trained_models = {}
histories = {}
summary_rows = []

for arm in ("raw_norm", "despeckled"):
    cfg = TrainConfig(
        image_arm=arm,
        input_height=128,
        input_width=192,
        base_channels=8,
        bottleneck_dropout=0.15,
        batch_size=16,
        epochs=18,
        learning_rate=2e-3,
        early_stopping_patience=7,
        seed=RANDOM_SEED,
    )
    model, hdf, ckpt_path = train_baseline_model(cfg)
    trained_models[arm] = model
    histories[arm] = hdf

    best_idx = int(hdf["val_dice"].idxmax())
    summary_rows.append(
        {
            "image_arm": arm,
            "best_epoch": int(hdf.loc[best_idx, "epoch"]),
            "best_val_dice_pct": round(float(hdf.loc[best_idx, "val_dice"]) * 100.0, 2),
            "best_val_iou_pct": round(float(hdf.loc[best_idx, "val_iou"]) * 100.0, 2),
            "best_val_loss": round(float(hdf.loc[best_idx, "val_loss"]), 5),
            "checkpoint": ckpt_path.name,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS_DIR / "baseline_training_summary.csv", index=False)
display(summary_df)

## 3. Learning Curves & Qualitative Validation Predictions
Compare convergence across both input arms and visually inspect segmentation overlays across 1st, 2nd, and 3rd trimester validation scans.

In [ ]:
curve_fig = plot_training_curves(histories)
pred_fig = plot_validation_predictions(trained_models, input_size=(128, 192))
display(Image(filename=str(curve_fig)))
display(Image(filename=str(pred_fig)))